In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns

kagglehub.login()

datathon_2026_round_1_path = kagglehub.competition_download('datathon-2026-round-1')
print('Data source import complete.')

data_dir = datathon_2026_round_1_path
dataframes = {}

for dirname, _, filenames in os.walk(data_dir):
    for filename in filenames:
        if filename.endswith('.csv'):
            path = os.path.join(dirname, filename)
            name = filename.replace('.csv', '')
            print(f"Loading: {name} from {path}")
            dataframes[name] = pd.read_csv(path)

print(len(dataframes))
if not dataframes:
    print("The dictionary is empty. Check if the file path is correct!")
else:
    print(f"Success! Found {len(dataframes)} files:")
    for name in dataframes.keys():
        print(f"- {name}")

print("--- Null Value Report ---")

for name, df in dataframes.items():
    null_counts = df.isnull().sum()
    missing_data = null_counts[null_counts > 0]
    print(f"\nDataFrame: {name}")
    if missing_data.empty:
        print("✅ No null values found!")
    else:
        print(missing_data)
        print(f"Total missing: {missing_data.sum()}")

dataframes['order_items']['promo_id'] = dataframes['order_items']['promo_id'].fillna(0)
dataframes['order_items']['promo_id_2'] = dataframes['order_items']['promo_id_2'].fillna(0)

print(dataframes['order_items'][['promo_id', 'promo_id_2']].isnull().sum())

for name, df in dataframes.items():
    print(f"--- {name} ---")
    print(df.dtypes)
    print("\n")

date_keywords = ['date', 'Date']

for name, df in dataframes.items():
    for col in df.columns:
        if any(key in col for key in date_keywords):
            print(f"Converting {name} -> {col}")
            df[col] = pd.to_datetime(df[col], errors='coerce')

print("\n--- DateTime conversion complete ---")

dataframes['promotions']['applicable_category'] = dataframes['promotions']['applicable_category'].fillna('All')
print(dataframes['promotions']['applicable_category'].value_counts())

print('--- Dtypes for order_items ---')
display(dataframes['order_items'].info())

import numpy as np

promo_map = dict(zip(dataframes['promotions']['promo_id'], dataframes['promotions']['applicable_category']))

dataframes['order_items']['applicable_category_1'] = dataframes['order_items']['promo_id'].map(promo_map)
dataframes['order_items']['applicable_category_2'] = dataframes['order_items']['promo_id_2'].map(promo_map)

dataframes['order_items']['applicable_category'] = dataframes['order_items']['applicable_category_1'].fillna(dataframes['order_items']['applicable_category_2'])
dataframes['order_items']['applicable_category'] = dataframes['order_items']['applicable_category'].fillna('None')

dataframes['order_items'] = dataframes['order_items'].drop(columns=['applicable_category_1', 'applicable_category_2'])

print(dataframes['order_items']['applicable_category'].value_counts())

print('--- Null Value Report After Cleaning ---')

for name, df in dataframes.items():
    if name in ['promotions', 'order_items']:
        null_counts = df.isnull().sum()
        missing_data = null_counts[null_counts > 0]
        print(f'\nDataFrame: {name}')
        if missing_data.empty:
            print('✅ No null values found!')
        else:
            print(missing_data)
            print(f'Total missing: {missing_data.sum()}')

print("\n### Basic descriptive statistics for each table")

for name, df in dataframes.items():
    print(f"\n--- DataFrame: {name} ---")
    print(f"Total records: {df.shape[0]}")
    print("\nNumber of unique values per column:")
    for col in df.columns:
        print(f"- {col}: {df[col].nunique()} unique values")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"\nDescriptive statistics for numeric column: {col}")
            print(df[col].describe())
        elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
            print(f"\nTop 5 most frequent values for column: {col}")
            print(df[col].value_counts().head())

output_dir = '/content/processed_data'
os.makedirs(output_dir, exist_ok=True)

for name, df in dataframes.items():
    file_path = os.path.join(output_dir, f'{name}_processed.csv')
    df.to_csv(file_path, index=False)
    print(f'Saved {name} to: {file_path}')

print(f'All DataFrames saved to: {output_dir}')

print("\n### Data Quality Check: COGS vs Price in products.csv")

df_products = dataframes['products']
issues_cogs_vs_price = df_products[df_products['cogs'] > df_products['price']]

if not issues_cogs_vs_price.empty:
    print("\nWarning: Products with COGS higher than Price:")
    print(issues_cogs_vs_price)
    print(f"\nTotal products with COGS > Price: {len(issues_cogs_vs_price)}")
else:
    print("\nNo products found with COGS higher than Price. ✅")

df_customers = dataframes['customers'].copy()
df_geography = dataframes['geography'].copy()
df_orders = dataframes['orders'].copy()
df_order_items = dataframes['order_items'].copy()
df_payments = dataframes['payments'].copy()

print("DataFrames loaded: df_customers, df_geography, df_orders, df_order_items, df_payments")

print("\n### Merging df_customers with df_geography")
df_customer_geo = pd.merge(df_customers, df_geography, on='zip', how='left')

print("df_customer_geo head:")
display(df_customer_geo.head())
print("\ndf_customer_geo info:")
df_customer_geo.info()

print("\n### Calculate total orders per customer")
total_orders_per_customer = df_orders.groupby('customer_id')['order_id'].count().reset_index()
total_orders_per_customer.rename(columns={'order_id': 'total_orders'}, inplace=True)

print("total_orders_per_customer head:")
display(total_orders_per_customer.head())

print("\n### Calculate total spending per customer")
df_orders_payments = pd.merge(df_orders, df_payments, on='order_id', how='left')
total_spending_per_customer = df_orders_payments.groupby('customer_id')['payment_value'].sum().reset_index()
total_spending_per_customer.rename(columns={'payment_value': 'total_spending'}, inplace=True)

print("total_spending_per_customer head:")
display(total_spending_per_customer.head())

print("\n### Find most recent purchase date per customer")
last_purchase_date_per_customer = df_orders.groupby('customer_id')['order_date'].max().reset_index()
last_purchase_date_per_customer.rename(columns={'order_date': 'last_purchase_date'}, inplace=True)

print("last_purchase_date_per_customer head:")
display(last_purchase_date_per_customer.head())

print("\n### Merge aggregated results into customer_data_mart")

customer_data_mart = pd.merge(df_customer_geo, total_orders_per_customer, on='customer_id', how='left')
customer_data_mart = pd.merge(customer_data_mart, total_spending_per_customer, on='customer_id', how='left')
customer_data_mart = pd.merge(customer_data_mart, last_purchase_date_per_customer, on='customer_id', how='left')

customer_data_mart['total_orders'] = customer_data_mart['total_orders'].fillna(0).astype(int)
customer_data_mart['total_spending'] = customer_data_mart['total_spending'].fillna(0)

print("customer_data_mart head:")
display(customer_data_mart.head())
print("\ncustomer_data_mart info:")
customer_data_mart.info()

df_products = dataframes['products'].copy()
df_inventory = dataframes['inventory'].copy()
df_returns = dataframes['returns'].copy()

print("DataFrames loaded: df_products, df_inventory, df_returns")

print("\n### Aggregate returns data by product")
total_returns_per_product = df_returns.groupby('product_id')['return_quantity'].sum().reset_index()
total_returns_per_product.rename(columns={'return_quantity': 'total_return_quantity'}, inplace=True)

print("total_returns_per_product head:")
display(total_returns_per_product.head())

print("\n### Merge products with inventory")
df_product_inventory = pd.merge(df_products, df_inventory, on='product_id', how='left')

print("df_product_inventory head:")
display(df_product_inventory.head())
print("\ndf_product_inventory info:")
df_product_inventory.info()

print("\n### Merge with aggregated returns")
df_product_inventory_returns = pd.merge(df_product_inventory, total_returns_per_product, on='product_id', how='left')

print("df_product_inventory_returns head:")
display(df_product_inventory_returns.head())

print("\n### Fill missing return quantities with 0")
product_data_mart = df_product_inventory_returns.copy()
product_data_mart['total_return_quantity'] = product_data_mart['total_return_quantity'].fillna(0).astype(int)

print("\n### Product Data Mart Summary")
print("product_data_mart head:")
display(product_data_mart.head())
print("\nproduct_data_mart info:")
product_data_mart.info()

print("\n### Calculate Gross Margin per product")
product_data_mart['gross_margin'] = np.where(
    product_data_mart['price'] != 0,
    (product_data_mart['price'] - product_data_mart['cogs']) / product_data_mart['price'],
    0
)

print("product_data_mart head with gross_margin:")
display(product_data_mart[['product_id', 'product_name_x', 'price', 'cogs', 'gross_margin']].head())

print("\n### Calculate total units sold per product")
total_units_sold_per_product = product_data_mart.groupby(['product_id', 'product_name_x', 'category_x', 'segment_x', 'size', 'color', 'price', 'cogs', 'total_return_quantity'])['units_sold'].sum().reset_index()
total_units_sold_per_product.rename(columns={'units_sold': 'total_units_sold'}, inplace=True)

print("total_units_sold_per_product head:")
display(total_units_sold_per_product.head())

print("\n### Merge total units sold into product_data_mart")
product_data_mart = pd.merge(product_data_mart, total_units_sold_per_product[['product_id', 'total_units_sold']], on='product_id', how='left')

print("product_data_mart head with total_units_sold:")
display(product_data_mart[['product_id', 'product_name_x', 'total_return_quantity', 'total_units_sold']].head())

print("\n### Calculate Return Rate per product")
product_data_mart['return_rate'] = np.where(
    product_data_mart['total_units_sold'] != 0,
    product_data_mart['total_return_quantity'] / product_data_mart['total_units_sold'],
    0
)

print("\n### Fill missing return rates with 0")
product_data_mart['return_rate'] = product_data_mart['return_rate'].fillna(0)

print("product_data_mart head with return_rate:")
display(product_data_mart[[
    'product_id', 'product_name_x', 'total_return_quantity', 'total_units_sold', 'return_rate'
]].head())

df_web_traffic = dataframes['web_traffic'].copy()
df_orders = dataframes['orders'].copy()

print("DataFrames loaded for Conversion Rate: df_web_traffic, df_orders")

print("\n### Aggregate web_traffic by traffic_source")
web_traffic_summary = df_web_traffic.groupby('traffic_source')[['sessions', 'unique_visitors']].sum().reset_index()

print("web_traffic_summary head:")
display(web_traffic_summary.head())

print("\n### Aggregate orders by order_source")
orders_summary = df_orders.groupby('order_source')['order_id'].count().reset_index()
orders_summary.rename(columns={'order_id': 'total_orders'}, inplace=True)

print("orders_summary head:")
display(orders_summary.head())

print("\n### Merge web traffic with orders")

orders_summary_renamed = orders_summary.rename(columns={'order_source': 'traffic_source'})
conversion_rate_df = pd.merge(web_traffic_summary, orders_summary_renamed, on='traffic_source', how='left')
conversion_rate_df['total_orders'] = conversion_rate_df['total_orders'].fillna(0).astype(int)

print("conversion_rate_df head:")
display(conversion_rate_df.head())
print("\nconversion_rate_df info:")
conversion_rate_df.info()

print("\n### Calculate Conversion Rate")
conversion_rate_df['conversion_rate'] = np.where(
    conversion_rate_df['sessions'] != 0,
    conversion_rate_df['total_orders'] / conversion_rate_df['sessions'],
    0
)

print("\n### Conversion Rate Results")
print("conversion_rate_df head:")
display(conversion_rate_df.head())
print("\nconversion_rate_df info:")
conversion_rate_df.info()

df_sales = dataframes['sales'].copy()
df_orders = dataframes['orders'].copy()
df_order_items = dataframes['order_items'].copy()
df_products = dataframes['products'].copy()
df_promotions = dataframes['promotions'].copy()
df_customers = dataframes['customers'].copy()
df_geography = dataframes['geography'].copy()

print("DataFrames loaded for Sales Data Mart")

print("\n### Merge orders with order_items")
df_orders_items = pd.merge(df_orders, df_order_items, on='order_id', how='left')

print("df_orders_items head:")
display(df_orders_items.head())
print("\ndf_orders_items info:")
df_orders_items.info()

print("\n### Calculate item total price after discount")
df_orders_items['item_total_price'] = (df_orders_items['quantity'] * df_orders_items['unit_price']) - df_orders_items['discount_amount']

print("df_orders_items head with item_total_price:")
display(df_orders_items.head())

print("\n### Calculate total order value per order_id")
df_total_order_value = df_orders_items.groupby('order_id')['item_total_price'].sum().reset_index()
df_total_order_value.rename(columns={'item_total_price': 'total_order_value'}, inplace=True)

print("df_total_order_value head:")
display(df_total_order_value.head())

print("\n### Merge orders_items with products")
df_orders_products = pd.merge(df_orders_items, df_products, on='product_id', how='left')

print("df_orders_products head:")
display(df_orders_products.head())
print("\ndf_orders_products info:")
df_orders_products.info()

print("\n### Merge with promotions details")

promotions_details_columns = [
    'promo_name', 'promo_type', 'discount_value', 'start_date', 'end_date',
    'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value'
]

df_promos_1_details = df_promotions[['promo_id'] + promotions_details_columns].copy()
df_promos_1_details = df_promos_1_details.rename(columns={
    col: f'{col}_promo_id1' for col in promotions_details_columns
}).rename(columns={'promo_id': 'promo_id_match_1'})

sales_data_mart = pd.merge(
    df_orders_products,
    df_promos_1_details,
    left_on='promo_id',
    right_on='promo_id_match_1',
    how='left'
)

df_promos_2_details = df_promotions[['promo_id'] + promotions_details_columns].copy()
df_promos_2_details = df_promos_2_details.rename(columns={
    col: f'{col}_promo_id2' for col in promotions_details_columns
}).rename(columns={'promo_id': 'promo_id_match_2'})

sales_data_mart = pd.merge(
    sales_data_mart,
    df_promos_2_details,
    left_on='promo_id_2',
    right_on='promo_id_match_2',
    how='left'
)

for col_base in promotions_details_columns:
    col_1 = f'{col_base}_promo_id1'
    col_2 = f'{col_base}_promo_id2'
    if col_1 in sales_data_mart.columns and col_2 in sales_data_mart.columns:
        sales_data_mart[col_1] = sales_data_mart[col_1].fillna(sales_data_mart[col_2])

cols_to_drop = [f'{col}_promo_id2' for col in promotions_details_columns] + \
               ['promo_id_match_1', 'promo_id_match_2']
sales_data_mart = sales_data_mart.drop(columns=cols_to_drop, errors='ignore')

sales_data_mart = sales_data_mart.rename(columns={
    f'{col}_promo_id1': col for col in promotions_details_columns
})

print("sales_data_mart head:")
display(sales_data_mart.head())
print("\nsales_data_mart info:")
sales_data_mart.info()

print("### Processing sales.csv for forecasting model")

df_sales_processed = dataframes['sales'].copy()
df_sales_processed['Date'] = pd.to_datetime(df_sales_processed['Date'])

min_date = df_sales_processed['Date'].min()
max_date = df_sales_processed['Date'].max()
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')

df_sales_processed = df_sales_processed.set_index('Date')
df_sales_processed = df_sales_processed.reindex(full_date_range, fill_value=0)

df_sales_processed['Revenue'] = df_sales_processed['Revenue'].fillna(0)
df_sales_processed['COGS'] = df_sales_processed['COGS'].fillna(0)

df_sales_processed = df_sales_processed.reset_index()
df_sales_processed.rename(columns={'index': 'Date'}, inplace=True)

print("\ndf_sales_processed head:")
display(df_sales_processed.head())
print("\ndf_sales_processed tail:")
display(df_sales_processed.tail())
print("\ndf_sales_processed info:")
df_sales_processed.info()

print("### Creating Lag features for df_sales_processed")

df_sales_processed = df_sales_processed.sort_values(by='Date').reset_index(drop=True)

df_sales_processed['Revenue_lag_1'] = df_sales_processed['Revenue'].shift(1)
df_sales_processed['Revenue_lag_7'] = df_sales_processed['Revenue'].shift(7)
df_sales_processed['COGS_lag_1'] = df_sales_processed['COGS'].shift(1)
df_sales_processed['COGS_lag_7'] = df_sales_processed['COGS'].shift(7)

df_sales_processed[['Revenue_lag_1', 'Revenue_lag_7', 'COGS_lag_1', 'COGS_lag_7']] = \
    df_sales_processed[['Revenue_lag_1', 'Revenue_lag_7', 'COGS_lag_1', 'COGS_lag_7']].fillna(0)

print("\ndf_sales_processed head with lag features:")
display(df_sales_processed.head())
print("\ndf_sales_processed info with lag features:")
df_sales_processed.info()

print('Loading Customer Data Mart...')
customer_data_mart_loaded = customer_data_mart.copy()
display(customer_data_mart_loaded.head())
print('Customer Data Mart loaded successfully.')

print('Loading Product Data Mart...')
product_data_mart_loaded = product_data_mart.copy()
display(product_data_mart_loaded.head())
print('Product Data Mart loaded successfully.')

print('Loading Sales Data Mart...')
sales_data_mart_loaded = sales_data_mart.copy()
display(sales_data_mart_loaded.head())
print('Sales Data Mart loaded successfully.')

print('Verifying Data Marts...')
if 'customer_data_mart' in locals() or 'customer_data_mart' in globals():
    customer_data_mart_loaded = customer_data_mart.copy()
    display(customer_data_mart_loaded.head())
    print('Customer Data Mart verified.')
else:
    print("Error: 'customer_data_mart' is not defined.")

if 'product_data_mart' in locals() or 'product_data_mart' in globals():
    product_data_mart_loaded = product_data_mart.copy()
    display(product_data_mart_loaded.head())
    print('Product Data Mart verified.')
else:
    print("Error: 'product_data_mart' is not defined.")

if 'sales_data_mart' in locals() or 'sales_data_mart' in globals():
    sales_data_mart_loaded = sales_data_mart.copy()
    display(sales_data_mart_loaded.head())
    print('Sales Data Mart verified.')
else:
    print("Error: 'sales_data_mart' is not defined.")

output_dir = '/content/processed_data'
os.makedirs(output_dir, exist_ok=True)

file_path_customer = os.path.join(output_dir, 'customer_data_mart.csv')
customer_data_mart.to_csv(file_path_customer, index=False)
print(f'Saved Customer Data Mart to: {file_path_customer}')

file_path_product = os.path.join(output_dir, 'product_data_mart.csv')
product_data_mart.to_csv(file_path_product, index=False)
print(f'Saved Product Data Mart to: {file_path_product}')

file_path_sales = os.path.join(output_dir, 'sales_data_mart.csv')
if 'sales_data_mart' in globals():
    sales_data_mart.to_csv(file_path_sales, index=False)
    print(f'Saved Sales Data Mart to: {file_path_sales}')
else:
    print("Warning: 'sales_data_mart' is not defined.")

print(f'All Data Marts saved to: {output_dir}')

df_order_items = dataframes['order_items'].copy()
df_orders = dataframes['orders'].copy()
df_products = dataframes['products'].copy()
df_customers = dataframes['customers'].copy()
df_geography = dataframes['geography'].copy()
df_payments = dataframes['payments'].copy()
df_shipments = dataframes['shipments'].copy()
df_promotions = dataframes['promotions'].copy()
df_inventory = dataframes['inventory'].copy()
df_returns = dataframes['returns'].copy()
df_reviews = dataframes['reviews'].copy()

print("All necessary DataFrames loaded.")

agg_returns = df_returns.groupby(['order_id', 'product_id']).agg(
    total_return_quantity=('return_quantity', 'sum'),
    total_refund_amount=('refund_amount', 'sum')
).reset_index()

agg_reviews = df_reviews.groupby(['order_id', 'product_id']).agg(
    avg_rating=('rating', 'mean'),
    review_count=('review_id', 'count')
).reset_index()

print("Aggregated returns and reviews data.")

df_master = pd.merge(df_order_items, df_orders,
                     on='order_id', how='left',
                     suffixes=('_item', '_order'))

print("Merged order_items with orders.")

df_master = pd.merge(df_master, df_products,
                     on='product_id', how='left',
                     suffixes=('_item', '_product'))

print("Merged with products.")

df_master = pd.merge(df_master, df_customers,
                     on='customer_id', how='left',
                     suffixes=('_order', '_customer'))

df_master = df_master.rename(columns={'zip_order': 'zip_code_customer'})
df_geography_renamed = df_geography.rename(columns={'zip': 'zip_code_customer'})
df_master = pd.merge(df_master, df_geography_renamed,
                     on='zip_code_customer', how='left',
                     suffixes=('_customer', '_geo'))

print("Merged with customers and geography.")

df_master = pd.merge(df_master, df_payments,
                     on='order_id', how='left',
                     suffixes=('', '_payment'))

df_master = pd.merge(df_master, df_shipments,
                     on='order_id', how='left',
                     suffixes=('', '_shipment'))

print("Merged with payments and shipments.")

promo_cols_to_merge = [
    'promo_name', 'promo_type', 'discount_value', 'start_date', 'end_date',
    'promo_channel', 'stackable_flag', 'min_order_value'
]

df_promotions_1 = df_promotions[['promo_id'] + promo_cols_to_merge].copy()
df_promotions_1.columns = ['promo_id'] + [f'promo1_{col}' for col in promo_cols_to_merge]

df_master = pd.merge(df_master, df_promotions_1, left_on='promo_id_item', right_on='promo_id', how='left')
df_master.drop(columns=['promo_id'], inplace=True)

df_promotions_2 = df_promotions[['promo_id'] + promo_cols_to_merge].copy()
df_promotions_2.columns = ['promo_id'] + [f'promo2_{col}' for col in promo_cols_to_merge]

df_master = pd.merge(df_master, df_promotions_2, left_on='promo_id_2', right_on='promo_id', how='left')
df_master.drop(columns=['promo_id'], inplace=True)

for col in promo_cols_to_merge:
    col1_name = f'promo1_{col}'
    col2_name = f'promo2_{col}'
    if col1_name in df_master.columns and col2_name in df_master.columns:
        df_master[col] = df_master[col1_name].fillna(df_master[col2_name])
        df_master.drop(columns=[col1_name, col2_name], inplace=True)

print("Merged with promotions.")

df_master['order_year_month'] = df_master['order_date'].dt.to_period('M')
df_inventory['inventory_year_month'] = df_inventory['snapshot_date'].dt.to_period('M')

df_inventory_clean = df_inventory.drop(columns=['snapshot_date', 'product_name', 'category', 'segment', 'year', 'month'])

df_master = pd.merge(df_master, df_inventory_clean,
                     left_on=['product_id', 'order_year_month'],
                     right_on=['product_id', 'inventory_year_month'],
                     how='left',
                     suffixes=('', '_inventory'))

df_master.drop(columns=['inventory_year_month', 'order_year_month'], inplace=True)

print("Merged with inventory.")

df_master = pd.merge(df_master, agg_returns,
                     on=['order_id', 'product_id'], how='left')

df_master = pd.merge(df_master, agg_reviews,
                     on=['order_id', 'product_id'], how='left')

print("Merged with returns and reviews.")

df_master['total_return_quantity'] = df_master['total_return_quantity'].fillna(0).astype(int)
df_master['total_refund_amount'] = df_master['total_refund_amount'].fillna(0)
df_master['avg_rating'] = df_master['avg_rating'].fillna(0)
df_master['review_count'] = df_master['review_count'].fillna(0).astype(int)

for col in ['stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate']:
    if col in df_master.columns:
        df_master[col] = df_master[col].fillna(0)

df_master['final_price'] = (df_master['quantity'] * df_master['unit_price']) - df_master['discount_amount']

print("Handled null values and calculated final_price.")

print(f"Final df_master shape: {df_master.shape}")
print("df_master head:")
display(df_master.head())
print("df_master info:")
df_master.info(verbose=True, show_counts=True)

print("### Re-creating df_sales_processed with lag features")

df_sales_processed = dataframes['sales'].copy()
df_sales_processed['Date'] = pd.to_datetime(df_sales_processed['Date'])
min_date = df_sales_processed['Date'].min()
max_date = df_sales_processed['Date'].max()
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')
df_sales_processed = df_sales_processed.set_index('Date')
df_sales_processed = df_sales_processed.reindex(full_date_range, fill_value=0)
df_sales_processed['Revenue'] = df_sales_processed['Revenue'].fillna(0)
df_sales_processed['COGS'] = df_sales_processed['COGS'].fillna(0)
df_sales_processed = df_sales_processed.reset_index()
df_sales_processed.rename(columns={'index': 'Date'}, inplace=True)

df_sales_processed = df_sales_processed.sort_values(by='Date').reset_index(drop=True)

df_sales_processed['Revenue_lag_1'] = df_sales_processed['Revenue'].shift(1)
df_sales_processed['Revenue_lag_7'] = df_sales_processed['Revenue'].shift(7)
df_sales_processed['COGS_lag_1'] = df_sales_processed['COGS'].shift(1)
df_sales_processed['COGS_lag_7'] = df_sales_processed['COGS'].shift(7)

df_sales_processed[['Revenue_lag_1', 'Revenue_lag_7', 'COGS_lag_1', 'COGS_lag_7']] = \
    df_sales_processed[['Revenue_lag_1', 'Revenue_lag_7', 'COGS_lag_1', 'COGS_lag_7']].fillna(0)

print("\ndf_sales_processed head with lag features:")
display(df_sales_processed.head())
print("\ndf_sales_processed info:")
df_sales_processed.info()

print("### Integrating promotional events into df_sales_processed")

df_promotions = dataframes['promotions'].copy()

promotion_dates = pd.DataFrame(columns=['Date', 'is_promotion_active'])

for index, row in df_promotions.iterrows():
    promo_date_range = pd.date_range(start=row['start_date'], end=row['end_date'], freq='D')
    temp_df = pd.DataFrame({'Date': promo_date_range, 'is_promotion_active': 1})
    promotion_dates = pd.concat([promotion_dates, temp_df])

promotion_dates = promotion_dates.drop_duplicates(subset=['Date']).reset_index(drop=True)

df_sales_processed = pd.merge(df_sales_processed,
                              promotion_dates,
                              on='Date',
                              how='left')

df_sales_processed['is_promotion_active'] = df_sales_processed['is_promotion_active'].fillna(0).astype(int)

print("\ndf_sales_processed head with promotion features:")
display(df_sales_processed.head())
print("\ndf_sales_processed info:")
df_sales_processed.info()

print("## Revenue Trend Analysis by Region (EDA)")

df_sales_with_region = pd.merge(
    sales_data_mart,
    customer_data_mart[['customer_id', 'region']],
    on='customer_id',
    how='left'
)

df_sales_with_region['order_day'] = df_sales_with_region['order_date'].dt.to_period('D')
revenue_by_region_daily = df_sales_with_region.groupby(['order_day', 'region'])['item_total_price'].sum().reset_index()
revenue_by_region_daily['order_day'] = revenue_by_region_daily['order_day'].dt.to_timestamp()

plt.figure(figsize=(15, 7))
sns.lineplot(data=revenue_by_region_daily, x='order_day', y='item_total_price', hue='region')
plt.title('Daily Revenue Trend by Region')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.grid(True)
plt.legend(title='Region')
plt.tight_layout()
plt.show()

print("### Key observations from revenue trend analysis:")
print("- Overall revenue trends and peak/low periods")
print("- Regional performance comparison")
print("- Identification of significant revenue fluctuations")

print("\n## Purchasing Behavior by Age Group (EDA)")

df_sales_with_demographics = pd.merge(
    sales_data_mart,
    customer_data_mart[['customer_id', 'age_group']],
    on='customer_id',
    how='left'
)

sales_by_age_group = df_sales_with_demographics.groupby('age_group')['item_total_price'].sum().reset_index()
sales_by_age_group = sales_by_age_group.sort_values(by='item_total_price', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='age_group', y='item_total_price', data=sales_by_age_group, palette='viridis')
plt.title('Total Revenue by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Total Revenue')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

print("### Key observations from age group analysis:")
print("- Top revenue-contributing age groups")
print("- Spending patterns across different age groups")

print("\n## Purchasing Behavior by Age Group and Channel (EDA)")

df_sales_with_demographics = pd.merge(
    sales_data_mart,
    customer_data_mart[['customer_id', 'age_group']],
    on='customer_id',
    how='left'
)

df_age_group_25_34 = df_sales_with_demographics[df_sales_with_demographics['age_group'] == '25-34'].copy()

purchasing_behavior_25_34 = df_age_group_25_34.groupby(['device_type', 'order_source']).agg(
    total_revenue=('item_total_price', 'sum'),
    total_orders=('order_id', 'nunique')
).reset_index()

plt.figure(figsize=(14, 7))
sns.barplot(x='order_source', y='total_revenue', hue='device_type', data=purchasing_behavior_25_34, palette='viridis')
plt.title('Total Revenue by Channel and Device for Age 25-34')
plt.xlabel('Order Source')
plt.ylabel('Total Revenue')
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.legend(title='Device')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 7))
sns.barplot(x='order_source', y='total_orders', hue='device_type', data=purchasing_behavior_25_34, palette='magma')
plt.title('Total Orders by Channel and Device for Age 25-34')
plt.xlabel('Order Source')
plt.ylabel('Total Orders')
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.legend(title='Device')
plt.tight_layout()
plt.show()

print("### Key observations from channel analysis:")
print("- Most popular channels and devices for age group 25-34")
print("- Revenue and order efficiency by channel-device combination")

print("\n## Summary: Data Analysis Key Findings\n")
print("### Data Quality Assessment:")
print("- Missing promo_id values filled with 0")
print("- Missing applicable_category filled with 'All'")
print("- No products with COGS > Price found")
print("\n### Data Mart Creation:")
print("- Customer Data Mart: 121,930 records")
print("- Product Data Mart: Complete product and inventory data")
print("- Sales Data Mart: 714,669 transaction records")
print("\n### KPI Engineering:")
print("- Gross Margin calculated per product")
print("- Return Rate calculated per product")
print("- Conversion Rate calculated per traffic source")
print("\n### Forecasting Preparation:")
print("- Continuous date range created (2012-2022)")
print("- Lag features generated for time-series analysis")
print("\n### EDA Insights:")
print("- Regional revenue trends analyzed")
print("- Age group purchasing patterns identified")
print("- Channel-device behavior analyzed for 25-34 age group")
